# EduGuard Risk Model Notebook
This notebook trains and evaluates a RandomForest model on `data/students_static.csv` with reproducible settings (`random_state=42`).

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Data Loading

In [ ]:
data_path = 'data/students_static.csv'

if not os.path.exists(data_path):
    raise FileNotFoundError(f'Dataset not found at: {data_path}')

df = pd.read_csv(data_path)
print('Dataset shape:', df.shape)
display(df.head())

## Preprocessing

In [ ]:
target_column = 'Dropout'
if target_column not in df.columns:
    raise ValueError(f"Target column '{target_column}' not found in dataset columns: {list(df.columns)}")

X = df.drop(columns=[target_column])
y = df[target_column]

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

## Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE
)

clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

clf.fit(X_train, y_train)
print('Model training complete.')

## Evaluation

In [ ]:
y_pred = clf.predict(X_test)

average_method = 'binary' if y.nunique() == 2 else 'weighted'

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average=average_method, zero_division=0)
recall = recall_score(y_test, y_pred, average=average_method, zero_division=0)
f1 = f1_score(y_test, y_pred, average=average_method, zero_division=0)

metrics_summary = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1': float(f1)
}

print('Accuracy :', round(accuracy, 4))
print('Precision:', round(precision, 4))
print('Recall   :', round(recall, 4))
print('F1-score :', round(f1, 4))

cm = confusion_matrix(y_test, y_pred)
report_text = classification_report(y_test, y_pred, zero_division=0)

print('\nClassification Report:\n')
print(report_text)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xlabel='Predicted label',
    ylabel='True label',
    title='Confusion Matrix'
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center', color='black')

plt.tight_layout()
plt.show()

## Saving Artifacts

In [ ]:
os.makedirs('metrics', exist_ok=True)
os.makedirs('models', exist_ok=True)

conf_matrix_path = 'metrics/confusion_matrix.png'
class_report_path = 'metrics/classification_report.txt'
metrics_json_path = 'metrics/metrics_summary.json'
model_path = 'models/risk_model.pkl'

fig.savefig(conf_matrix_path, dpi=300, bbox_inches='tight')

with open(class_report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

with open(metrics_json_path, 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, indent=2)

joblib.dump(clf, model_path)

print('Saved:', conf_matrix_path)
print('Saved:', class_report_path)
print('Saved:', metrics_json_path)
print('Saved:', model_path)